# Funções

  Para desenvolver o módulo de funções, foram importadas funções e constantes dos módulos constantes.py e aritmetica.py para serem usados como base para os novos algoritmos mais avançados deste módulo. Além disso, algumas fuções fizeram importações adicionais baseadas nos seus propósitos.

  Com isso, foram desenvolvidos algoritmos para calcular exponencial, logaritmo (natural, base 10, base 2 e de base a escolha do usuário) e raízes quadrada e cúbica, essas com maior precisão do que os algoritmos desenvolvidos no módulo aritmética, e uma função que calcula a hipotenusa de triângulos. 
  
  Também foram desenvolvidos algoritmos que realizam funções especiais, e as escolhidas por mim foram a função gamma, uma função para calcular o fatorial de números reais com base na função gamma e a função lgamma, que calcula o logaritmo natural da função gamma; a função beta, a função beta incompleta e a função beta incompleta regularizada; a função de erro de Gauss, a função de erro complementar e a inversa da função de erro; a função de Bessel de 1° ordem inteira, a função de Bessel de 2° ordem inteira, a função de Bessel modificada de 1° espécie e a função de Bessel modificada de 2° espécie; as funções de fatorial decrescente e crescente como auxiliares para a série de Borwein, uma função para calcular os coeficientes dₖ de Borwein para acelerar a série da função eta de Dirichlet, a função eta de Dirichlet, baseada nas funções anteriores, e zeta de Riemman, que usa como base a própria função eta de Dirichlet; uma função que calcula o símbolo de Pochhammer, que serve como base para as funções hipergeométricas, e as funções hipergeométricas confluente, confluente de Kummer e de Gauss.

  Para gerar essas funções foi usado como base a lógica matemática por trás dessas funções. Para isso, o código das funções especiais foi baseado em módulos externos já existentes, tanto pela complexidade, quanto pelo não domínio dos conhecimentos matemáticos que envolvem as mesmas.

  Para finalizar, adicionou-se a linha de código "%%writefile funcoes.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile funcoes.py

from constantes import (pi, e, tolerancia_de_erro, maximo_de_interacoes, inf)
from aritmetica import (potencia_rapida_int, piso, valor_absoluto, ln_local)

LN2 = ln_local(2)

#Exponencial
def exponencial(x: float) -> float:

    if x > 709.78:
        return inf
    if x < -745.13:
        return 0.0
    if x == 0:
        return 1.0

    n = int(x / LN2 + 0.5)
    r = x - n * LN2

    termo = 1.0
    resultado = 1.0
    for k in range(1, maximo_de_interacoes):
        termo *= r / k
        resultado += termo
        if valor_absoluto(termo) < tolerancia_de_erro * 1e-10:
            break

    if n >= 0:
        return resultado * potencia_rapida_int(2.0, n)
    else:
        return resultado / potencia_rapida_int(2.0, -n)


#Logaritmos
#Logaritmo natural (ln)
def ln(x: float) -> float:

    if x <= 0:
        raise ValueError("ln(x) é indefinido para x <= 0.")
    if x == 1:
        return 0.0
    if x == e:
        return 1.0

    k = 0
    m = x
    while m >= 2.0:
        m /= 2.0
        k += 1
    while m < 0.5:
        m *= 2.0
        k -= 1

    y = (m - 1.0) / (m + 1.0)
    y2 = y * y
    termo = y
    resultado = 0.0
    for n in range(maximo_de_interacoes):
        resultado += termo / (2 * n + 1)
        termo *= y2
        if valor_absoluto(termo) / (2 * n + 3) < tolerancia_de_erro * 1e-10:
            break

    return 2.0 * resultado + k * LN2

#Logaritmo na base 10
def log10(x: float) -> float:

    if x <= 0:
        raise ValueError("log10(x) é indefinido para x <= 0.")
    return ln(x) / ln(10)

#Logaritmo na base 2
def log2(x: float) -> float:

    if x <= 0:
        raise ValueError("log2(x) é indefinido para x <= 0.")
    return ln(x) / ln(2)

#Logaritmo em qualquer base
def log_base(x: float, base: float) -> float:

    if x <= 0:
        raise ValueError("log_b(x) é indefinido para x <= 0.")
    if base <= 0 or base == 1:
        raise ValueError("Base de logaritmo inválida, deve ser > 0 e != 1.")
    return ln(x) / ln(base)


#Raízes
#Raíz quadrada
def sqrt(x: float) -> float:

    if x < 0:
        raise ValueError("Raiz quadrada de número negativo é complexa (usar funções complexas).")
    if x == 0:
        return 0.0
    if x == 1:
        return 1.0

    estimativa = x if x < 1 else x / 2.0

    for _ in range(maximo_de_interacoes):
        nova_estimativa = (estimativa + x / estimativa) / 2.0
        if valor_absoluto(nova_estimativa - estimativa) < tolerancia_de_erro * 1e-8:
            return nova_estimativa
        estimativa = nova_estimativa

    return estimativa

#Raíz cúbica
def cbrt(x: float) -> float:

    if x == 0:
        return 0.0
    sinal_x = 1 if x > 0 else -1
    x = valor_absoluto(x)
    estimativa = x / 3.0 if x > 1 else x + 0.1

    for _ in range(maximo_de_interacoes):
        g2 = estimativa * estimativa
        if g2 == 0:
            break
        nova_estimativa = (2.0 * estimativa + x / g2) / 3.0
        if valor_absoluto(nova_estimativa - estimativa) < tolerancia_de_erro * 1e-8:
            return sinal_x * nova_estimativa
        estimativa = nova_estimativa

    return sinal_x * estimativa


#Função Gamma
def gamma(x: float) -> float:

    LANCZOS_COEFF = [
        0.99999999999980993,
        676.5203681218851,
        -1259.1392167224028,
        771.32342877765313,
        -176.61502916214059,
        12.507343278686905,
        -0.13857109526572012,
        9.9843695780195716e-6,
        1.5056327351493116e-7,
    ]

    if x <= 0 and x == piso(x):
        raise ValueError("A função gamma(x) é indefinida para inteiros não-positivos.")

    if x < 0.5:
        from trigonometria import sin
        return pi / (sin(pi * x) * gamma(1.0 - x))

    x -= 1.0
    a = LANCZOS_COEFF[0]
    t = x + 7.5

    for i in range(1, 9):
        a += LANCZOS_COEFF[i] / (x + i)
        
    return (sqrt(2.0 * pi) * exponencial((x + 0.5) * ln(t))* exponencial(-t) * a)

#Fatorial para números reais pela função Gamma
def fatorial_real(x: float) -> float:

    if x == piso(x) and x >= 0:
        from aritmetica import fatorial
        return float(fatorial(int(x)))
    return gamma(x + 1.0)


#Logaritmo natural do valor absoluto da função Gamma
def lgamma(x: float) -> float:

    LANCZOS_COEFF = [
        0.99999999999980993,  676.5203681218851,   -1259.1392167224028,
        771.32342877765313,  -176.61502916214059,    12.507343278686905,
        -0.13857109526572012,  9.9843695780195716e-6, 1.5056327351493116e-7,
    ]

    if x <= 0 and x == piso(x):
        raise ValueError("A função lgamma(x) é indefinida para inteiros não-positivos.")

    if x < 0.5:
        from trigonometria import sin
        return ln(pi) - ln(valor_absoluto(sin(pi * x))) - lgamma(1.0 - x)

    x -= 1.0
    a = LANCZOS_COEFF[0]
    t = x + 7.5
    for i in range(1, 9):
        a += LANCZOS_COEFF[i] / (x + i)

    return 0.5 * ln(2.0 * pi) + (x + 0.5) * ln(t) - t + ln(valor_absoluto(a))


#Função Beta
def beta(a: float, b: float) -> float:

    if a <= 0 or b <= 0:
        raise ValueError("A função beta deve ser executada com dois números maiores que 0.")
    return exponencial(lgamma(a) + lgamma(b) - lgamma(a + b))


#Função Beta incompleta
def beta_inc(x: float, a: float, b: float) -> float:

    if not (0.0 <= x <= 1.0):
        raise ValueError("A função beta incompleta deve ser executada com três números e o primeiro deles no intervalo [0, 1].")
    if a <= 0 or b <= 0:
        raise ValueError("A função beta incompleta deve ser executada com três números e o primeiro deles no intervalo [0, 1].")
    if x == 0.0:
        return 0.0
    if x == 1.0:
        return beta(a, b)

    if x > (a + 1.0) / (a + b + 2.0):
        return beta(a, b) - beta_inc(1.0 - x, b, a)

    log_front = a * ln(x) + b * ln(1.0 - x) - ln(a) - lgamma(a) - lgamma(b) + lgamma(a + b)
    front = exponencial(log_front)

    fpmin = 1e-300
    qab = a + b
    qap = a + 1.0
    qam = a - 1.0
    c = 1.0
    d = 1.0 - qab * x / qap
    if valor_absoluto(d) < fpmin:
        d = fpmin
    d = 1.0 / d
    h = d

    for m in range(1, maximo_de_interacoes + 1):
        m2 = 2 * m

        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if valor_absoluto(d) < fpmin:
            d = fpmin
        c = 1.0 + aa / c
        if valor_absoluto(c) < fpmin:
            c = fpmin
        d = 1.0 / d
        h *= d * c

        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if valor_absoluto(d) < fpmin:
            d = fpmin
        c = 1.0 + aa / c
        if valor_absoluto(c) < fpmin:
            c = fpmin
        d = 1.0 / d
        delta = d * c
        h *= delta

        if valor_absoluto(delta - 1.0) < tolerancia_de_erro:
            break

    return front * h


#Função Beta incompleta regularizada
def beta_reg(x: float, a: float, b: float) -> float:

    return beta_inc(x, a, b) / beta(a, b)


#Função de erro de Gauss
def erf(x: float) -> float:

    if x == 0.0:
        return 0.0
    if valor_absoluto(x) > 6.0:
        return 1.0 if x > 0 else -1.0

    resultado = 0.0
    termo = x
    x2 = x * x

    for k in range(maximo_de_interacoes):
        resultado += termo / (2 * k + 1)
        proximo_termo = -termo * x2 / (k + 1)
        if valor_absoluto(proximo_termo) < tolerancia_de_erro * 1e-10:
            break
        termo = proximo_termo

    constante = 2.0 / sqrt(pi)
    return constante * resultado


#Função de erro complementar
def erfc(x: float) -> float:

    if x > 4.0:
        z = x * x
        termo = 1.0
        resultado = 1.0
        for k in range(1, 20):
            termo *= -(2 * k - 1) / (2.0 * z)
            resultado += termo
            if valor_absoluto(termo) < tolerancia_de_erro:
                break
        return (exponencial(-z) / (x * sqrt(pi))) * resultado

    if x < -4.0:
        return 2.0 - erfc(-x)

    return 1.0 - erf(x)


#Inversa da função de erro
def erf_inv(p: float, tol: float = tolerancia_de_erro) -> float:

    if valor_absoluto(p) >= 1.0:
        raise ValueError("A inversa da função de erro deve ser executada em um valor pertencente ao intervalo (−1, 1).")
    if p == 0.0:
        return 0.0

    a = 0.147
    ln1p2 = ln(1.0 - p * p)
    termo1 = 2.0 / (pi * a) + ln1p2 / 2.0
    x = (1 if p > 0 else -1) * sqrt(sqrt(termo1 * termo1 - ln1p2 / a) - termo1)

    constante = 2.0 / sqrt(pi)
    for _ in range(maximo_de_interacoes):
        fx = erf(x) - p
        fpx = constante * exponencial(-x * x)
        if valor_absoluto(fpx) < 1e-300:
            break
        dx = fx / fpx
        x -= dx
        if valor_absoluto(dx) < tol:
            break

    return x


#Função de Bessel de 1ª espécie de ordem inteira n: J_n(x).
def bessel_j(n: int, x: float) -> float:

    if n < 0:
        sinal_n = 1 if (-n) % 2 == 0 else -1
        return sinal_n * bessel_j(-n, x)
    if x == 0.0:
        return 1.0 if n == 0 else 0.0

    metade_x = x / 2.0
    from aritmetica import fatorial

    termo = potencia_rapida_int(metade_x, n) / fatorial(n)
    resultado = termo
    metade_x_sq = -metade_x * metade_x

    for k in range(1, maximo_de_interacoes):
        termo *= metade_x_sq / (k * (n + k))
        resultado += termo
        if valor_absoluto(termo) < valor_absoluto(resultado) * tolerancia_de_erro * 1e-8:
            break

    return resultado


#Função de Bessel de 2ª espécie (Neumann) de ordem inteira n: Y_n(x).
def bessel_y(n: int, x: float) -> float:

    if x <= 0:
        raise ValueError("A função de Bessel de 2° espécie é indefinida quando aplicada com segundo valor menor ou igual a 0.")

    EULER_MASCHERONI = 0.5772156649015328

    def y0(x):
        j0 = bessel_j(0, x)

        metade_x = x / 2.0
        metade_x_sq = -metade_x * metade_x
        termo = 1.0
        serie = 0.0
        H = 0.0
        from aritmetica import fatorial
        for k in range(1, maximo_de_interacoes):
            termo *= metade_x_sq / (k * k)
            H += 1.0 / k
            serie += H * termo
            if valor_absoluto(termo * H) < tolerancia_de_erro * 1e-8:
                break
        return (2.0 / pi) * (j0 * (ln(x / 2.0) + EULER_MASCHERONI) + serie)

    def y1(x):
        j1 = bessel_j(1, x)
        metade_x = x / 2.0
        metade_x_sq = -metade_x * metade_x

        termo = metade_x
        serie = 0.0
        H = 1.0
        from aritmetica import fatorial
        for k in range(1, maximo_de_interacoes):
            termo *= metade_x_sq / (k * (k + 1))
            H += 1.0 / (k + 1) + 1.0 / k
            serie += H * termo
            if valor_absoluto(termo * H) < tolerancia_de_erro * 1e-8:
                break
        return (2.0 / pi) * (j1 * (ln(x / 2.0) + EULER_MASCHERONI) - 1.0 / x - serie)

    if n == 0:
        return y0(x)
    if n == 1:
        return y1(x)

    y_anterior = y0(x)
    y_atual = y1(x)
    for k in range(1, n):
        y_seguinte = (2.0 * k / x) * y_atual - y_anterior
        y_anterior = y_atual
        y_atual = y_seguinte

    return y_atual


#Função de Bessel modificada de 1ª espécie: I_n(x).
def bessel_i(n: int, x: float) -> float:

    from aritmetica import fatorial
    if x == 0.0:
        return 1.0 if n == 0 else 0.0
    if n < 0:
        return bessel_i(-n, x)

    metade_x = x / 2.0
    termo = potencia_rapida_int(metade_x, n) / fatorial(n)
    resultado = termo
    metade_x_sq = metade_x * metade_x

    for k in range(1, maximo_de_interacoes):
        termo *= metade_x_sq / (k * (n + k))
        resultado += termo
        if valor_absoluto(termo) < valor_absoluto(resultado) * tolerancia_de_erro * 1e-8:
            break

    return resultado


#Função de Bessel modificada de 2ª espécie: K_n(x).
def bessel_k(n: int, x: float) -> float:

    if x <= 0:
        raise ValueError("A função de Bessel modificada de 2° espécie é indefinida quando aplicada com segundo valor menor ou igual a 0.")

    EULER_MASCHERONI = 0.5772156649015328

    def k0(x):
        i0 = bessel_i(0, x)
        metade_x = x / 2.0
        metade_x_sq = metade_x * metade_x
        termo = 1.0
        serie = 0.0
        H = 0.0
        from aritmetica import fatorial
        for k in range(1, maximo_de_interacoes):
            termo *= metade_x_sq / (k * k)
            H += 1.0 / k
            serie += H * termo
            if valor_absoluto(termo * H) < tolerancia_de_erro * 1e-8:
                break
        return -(ln(x / 2.0) + EULER_MASCHERONI) * i0 + serie

    def k1(x):
        i1 = bessel_i(1, x)
        metade_x = x / 2.0
        metade_x_sq = metade_x * metade_x
        termo = metade_x
        serie = 0.0
        H = 1.0
        from aritmetica import fatorial
        for k in range(1, maximo_de_interacoes):
            termo *= metade_x_sq / (k * (k + 1))
            H += 1.0 / k + 1.0 / (k + 1)
            serie += H * termo
            if valor_absoluto(termo * H) < tolerancia_de_erro * 1e-8:
                break
        return (1.0 / x) + (ln(x / 2.0) + EULER_MASCHERONI) * i1 - serie

    if n == 0:
        return k0(x)
    if n == 1:
        return k1(x)

    k_anterior = k0(x)
    k_atual = k1(x)
    for m in range(1, n):
        k_seguinte = (2.0 * m / x) * k_atual + k_anterior
        k_anterior = k_atual
        k_atual = k_seguinte

    return k_atual


#Função Zeta de Riemann
#Auxiliares para a série de Borwein
def fatorial_decrescente(x: float, k: int) -> float:

    r = 1.0
    for i in range(k):
        r *= (x - i)
    return r


def fatorial_crescente(x: float, k: int) -> float:

    r = 1.0
    for i in range(k):
        r *= (x + i)
    return r


#Coeficientes de Borwein para aceleração da série eta.
def borwein_coeffs(n: int) -> list:

    d = [0.0] * (n + 1)
    d[0] = n
    for k in range(1, n + 1):
        d[k] = d[k - 1] * (n - k + 1) * (2 * k - 1) / (k * (2 * (n - k) + 1))
    for k in range(1, n + 1):
        d[k] += d[k - 1]
    return d


#Função Eta de Dirichlet
def dirichlet_eta(s: float) -> float:

    n = 60
    d = borwein_coeffs(n)

    resultado = 0.0
    dn = d[n]
    for k in range(n):
        sinal = 1.0 if k % 2 == 0 else -1.0
        resultado += sinal * (d[k] - dn) / exponencial(s * ln_local(k + 1))

    return -resultado / dn


#Potência de base 2: 2^x = e^(x·ln2)
def exponencial_2(x: float) -> float:
    return exponencial(x * ln_local(2))


def zeta(s: float) -> float:

    if s == 1.0:
        raise ValueError("A função zeta de 1 é indefinida, pois é polo simples de singularidade.")

    if s < 0 and s == piso(s) and int(-s) % 2 == 0:
        return 0.0

    pi2 = pi * pi
    exato = {
        2:  pi2 / 6.0,
        4:  pi2 * pi2 / 90.0,
        6:  pi2 * pi2 * pi2 / 945.0,
        8:  pi2 * pi2 * pi2 * pi2 / 9450.0,
        10: pi2 * pi2 * pi2 * pi2 * pi2 / 93555.0,
    }
    if s in exato:
        return exato[s]
    if s == 0:
        return -0.5
    if s == -1:
        return -1.0 / 12.0

    eta = dirichlet_eta(s)
    fator = 1.0 - exponencial_2(1.0 - s)

    if valor_absoluto(fator) < tolerancia_de_erro:
        EULER_MASCHERONI = 0.5772156649015328
        return 1.0 / (s - 1.0) + EULER_MASCHERONI

    return eta / fator


#Zeta via série alternada direta
def zeta_alt(s: float, termos: int = 200) -> float:

    if s == 1.0:
        raise ValueError("A função zeta alternada de 1 é indefinida, pois é polo simples de singularidade.")
    eta = 0.0
    sinal = 1.0
    for n in range(1, termos + 1):
        eta += sinal / exponencial(s * ln_local(n))
        sinal = -sinal
    denominador = 1.0 - exponencial_2(1.0 - s)
    if valor_absoluto(denominador) < tolerancia_de_erro:
        return float("inf")
    return eta / denominador


#Funções hipergeométricas
#Símbolo de Pochhammer
def pochhammer(a: float, n: int) -> float:
    if n == 0:
        return 1.0
    resultado = 1.0
    for k in range(n):
        resultado *= (a + k)
    return resultado


#Função hipergeométrica confluente ₀F₁(; b; z)
def hyp0f1(b: float, z: float) -> float:

    if b <= 0 and b == piso(b):
        raise ValueError("A função hipergeométrica confluente não pode ser executada com primeiro valor sendo inteiro não-positivo.")

    resultado = 0.0
    termo = 1.0

    for n in range(maximo_de_interacoes):
        resultado += termo
        proximo_termo = termo * z / ((b + n) * (n + 1))
        if valor_absoluto(proximo_termo) < valor_absoluto(resultado) * tolerancia_de_erro * 1e-10:
            break
        termo = proximo_termo

    return resultado


#Função hipergeométrica confluente de Kummer ₁F₁(a; b; z) / M(a, b, z)
def hyp1f1(a: float, b: float, z: float) -> float:

    if b <= 0 and b == piso(b):
        raise ValueError("A função hipergeométrica confluente de Kummer não pode ser executada com segundo valor sendo inteiro não-positivo.")

    if z > 50:
        return exponencial(z) * hyp1f1(b - a, b, -z)

    resultado = 0.0
    termo = 1.0

    for n in range(maximo_de_interacoes):
        resultado += termo

        proximo_termo = termo * (a + n) * z / ((b + n) * (n + 1))
        if valor_absoluto(proximo_termo) < valor_absoluto(resultado) * tolerancia_de_erro * 1e-10:
            break
        if valor_absoluto(proximo_termo) > 1e200:
            break
        termo = proximo_termo

    return resultado


#Função hipergeométrica de Gauss ₂F₁(a, b; c; z)
def hyp2f1(a: float, b: float, c: float, z: float) -> float:

    if c <= 0 and c == piso(c):
        raise ValueError("A função hipergeométrica de Gauss não pode ser executada com terceiro valor sendo inteiro não-positivo.")

    if z == 0.0:
        return 1.0
    if z == 1.0:
        if c - a - b <= 0:
            raise ValueError("A função hipergeométrica de Gauss diverge pois c <= a + b.")
        return gamma(c) * gamma(c - a - b) / (gamma(c - a) * gamma(c - b))

    if valor_absoluto(z) > 0.5 and valor_absoluto(z) < 1.0:
        w = z / (z - 1.0)
        return exponencial(-a * ln(1.0 - z)) * hyp2f1(a, c - b, c, w)

    if valor_absoluto(z) >= 1.0:
        if z < -1.0:
            w = z / (z - 1.0)
            return exponencial(-a * ln(1.0 - z)) * hyp2f1(a, c - b, c, w)
        raise ValueError("A função hipergeométrica de Gauss possui quarto valor fora do domínio de convergência (|z| < 1).")

    resultado = 0.0
    termo = 1.0

    for n in range(maximo_de_interacoes):
        resultado += termo
        proximo_termo = termo * (a + n) * (b + n) * z / ((c + n) * (n + 1))
        if valor_absoluto(proximo_termo) < valor_absoluto(resultado) * tolerancia_de_erro * 1e-10:
            break
        termo = proximo_termo

    return resultado

Overwriting funcoes.py
